[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module1/5_Appendix_ROC_AUC_and_Thresholds.ipynb)

# Appendix - ROC, AUC, and the Decision Threshold

**OPIM 5509: Introduction to Deep Learning - University of Connecticut**

This is an appendix, not a lecture. There is no video. Work through it when the ROC curve in `3_AllTheModels_Classification.ipynb` felt like a plot that appeared out of nowhere.

Here is the claim this notebook is going to make concrete:

> **A classifier does not make decisions. You do.** The model hands you a probability. You pick the number that turns that probability into a yes or a no - and that number is a *business* choice, not a technical one.

We are going to build an ROC curve **by hand** on twelve rows of data you can check with a pencil, watch what the threshold does to the false positive rate, and only then let scikit-learn draw the same curve. Doing it by hand first is the whole point - the curve stops being decoration and becomes something you understand well enough to argue with.

🔷 **The nugget:** the ROC curve is not a property of your predictions. It is what you get when you slide the threshold from 1.0 down to 0.0 and record where the confusion matrix lands at every stop.

## 1. Twelve loan applicants

A model has scored twelve applicants for the probability they will **default**. We also know, after the fact, who actually did.

Two things can go wrong, and they cost different amounts:

| Mistake | What it means here | What it costs |
| :-- | :-- | :-- |
| **False positive** | Flag a customer who would have paid you back | You lose a good customer and the interest they'd have paid |
| **False negative** | Approve a customer who defaults | You lose the principal |

Hold that asymmetry in mind. It is the whole reason the threshold matters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GOLD = "#C99300"

# Twelve applicants, sorted from riskiest-looking to safest-looking.
loans = pd.DataFrame({
    "applicant": list("ABCDEFGHIJKL"),
    "score":  [0.95, 0.90, 0.85, 0.80, 0.72, 0.65, 0.60, 0.55, 0.45, 0.40, 0.30, 0.15],
    "actual": [   1,    1,    0,    1,    1,    0,    1,    0,    1,    0,    0,    0],
})

print("Positives (actually defaulted):", loans["actual"].sum())
print("Negatives (paid it back)      :", (1 - loans["actual"]).sum())
loans

Notice applicant **C**: the model gave them a 0.85 - the third-highest risk score in the pool - and they paid the loan back just fine. The model is *good*, not perfect. If it were perfect every 1 would sit above every 0 and there would be nothing to teach.

## 2. One threshold, one confusion matrix

Pick a threshold. Everyone at or above it gets flagged. Everyone below is approved.

Start at **0.50** - the default `predict()` uses, and a number with no special meaning beyond being halfway.

In [ ]:
def confusion_at(df, threshold):
    """Return TP, FP, FN, TN for one threshold."""
    flagged = df["score"] >= threshold
    tp = int(((df["actual"] == 1) & flagged).sum())
    fp = int(((df["actual"] == 0) & flagged).sum())
    fn = int(((df["actual"] == 1) & ~flagged).sum())
    tn = int(((df["actual"] == 0) & ~flagged).sum())
    return tp, fp, fn, tn


tp, fp, fn, tn = confusion_at(loans, 0.50)

print("Threshold = 0.50")
print()
print("                  predicted APPROVE   predicted FLAG")
print(f"  actually paid        TN = {tn:<12} FP = {fp}")
print(f"  actually defaulted   FN = {fn:<12} TP = {tp}")

### The two rates that build the curve

Everything about an ROC curve comes from exactly two numbers, and both are computed **down the rows** of that table - each one uses only one of the two actual classes:

$$\text{TPR} = \frac{TP}{TP + FN} = \frac{\text{defaulters we caught}}{\text{all defaulters}}$$

$$\text{FPR} = \frac{FP}{FP + TN} = \frac{\text{good customers we wrongly flagged}}{\text{all good customers}}$$

**TPR** is also called *recall* or *sensitivity*. **FPR** is the one Dave wants you to watch: it is the fraction of your *innocent* cases that get caught in the net.

**Remember:** TPR uses only the actual-positive row. FPR uses only the actual-negative row. That separation is why ROC curves behave the way they do when classes are imbalanced - more on that in section 8.

In [ ]:
def rates_at(df, threshold):
    tp, fp, fn, tn = confusion_at(df, threshold)
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return tpr, fpr


tpr, fpr = rates_at(loans, 0.50)
print(f"At threshold 0.50:  TPR = {tpr:.3f}   FPR = {fpr:.3f}")
print()
print(f"  We catch {tpr:.0%} of the people who actually defaulted,")
print(f"  and we wrongly flag {fpr:.0%} of the people who would have paid.")

## 3. Now move the threshold

This is the section that matters. Watch **FPR** as the threshold comes down.

In [ ]:
rows = []
for t in [1.00, 0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00]:
    tp, fp, fn, tn = confusion_at(loans, t)
    tpr, fpr = rates_at(loans, t)
    rows.append({"threshold": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "TPR (caught)": round(tpr, 3), "FPR (false alarms)": round(fpr, 3)})

sweep = pd.DataFrame(rows).set_index("threshold")
sweep

Read that table top to bottom and say what is happening in plain language:

- **Threshold 1.00** - flag nobody. You catch zero defaulters (TPR = 0) but you never falsely accuse anyone (FPR = 0). Perfectly safe, perfectly useless.
- **Coming down** - you start catching real defaulters. Every so often you also snag someone who would have paid.
- **Threshold 0.00** - flag everybody. You catch every single defaulter (TPR = 1.0) and you also flag every single good customer (FPR = 1.0). Also useless, in the opposite direction.

**There is no threshold that gives you a high TPR and a low FPR for free.** Lowering the threshold to catch more defaulters *always* drags more good customers in with them. That trade is not a flaw in the model. It is the shape of the problem.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(sweep.index, sweep["TPR (caught)"], "o-", color=GOLD, lw=2,
        label="TPR - defaulters caught")
ax.plot(sweep.index, sweep["FPR (false alarms)"], "s--", color="#3D2E00", lw=2,
        label="FPR - good customers wrongly flagged")

ax.set_xlabel("Decision threshold")
ax.set_ylabel("Rate")
ax.set_title("Lower the threshold and BOTH rates rise - that is the whole trade")
ax.invert_xaxis()          # read left-to-right as "getting more aggressive"
ax.legend()
plt.tight_layout()
plt.show()

## 4. The ROC curve is that table, replotted

An ROC curve throws the threshold away as an axis and plots **FPR on x against TPR on y**. Each point is one threshold. Connect them and you have the curve.

That is genuinely all it is.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.plot(sweep["FPR (false alarms)"], sweep["TPR (caught)"],
        "o-", color=GOLD, lw=2, markersize=7)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="coin flip")

# label a few points with the threshold that produced them
for t in [0.90, 0.60, 0.40, 0.10]:
    x, y = sweep.loc[t, "FPR (false alarms)"], sweep.loc[t, "TPR (caught)"]
    ax.annotate(f"  t={t:.2f}", (x, y), fontsize=9, va="center")

ax.set_xlabel("False positive rate  (good customers wrongly flagged)")
ax.set_ylabel("True positive rate  (defaulters caught)")
ax.set_title("ROC curve - built by hand from twelve rows")
ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**How to read any ROC curve:**

- **Bottom-left (0, 0)** - the paranoid corner. Flag nobody, catch nobody, accuse nobody.
- **Top-right (1, 1)** - the panic corner. Flag everybody, catch everybody, accuse everybody.
- **Top-left (0, 1)** - perfection. Catch every defaulter, never flag a good customer. No real model gets here.
- **The diagonal** - a coin flip. A model on this line has learned nothing; its scores carry no information about who defaults.

**The further your curve bulges toward the top-left, the better the model separates the two classes.**

## 5. Every threshold, not just the round numbers

Round thresholds like 0.7 and 0.6 are arbitrary. The curve only actually changes when the threshold crosses one of the twelve **scores** - between two scores nothing moves, because no applicant changes sides.

So the honest way to build it is to use every observed score as a threshold. That gives the staircase shape a real ROC curve has.

In [ ]:
# Every score is a candidate threshold, plus one above the maximum (flag nobody).
thresholds = [1.01] + sorted(loans["score"], reverse=True)

pts = []
for t in thresholds:
    tp, fp, fn, tn = confusion_at(loans, t)
    tpr, fpr = rates_at(loans, t)
    pts.append({"threshold": t, "TP": tp, "FP": fp, "TPR": tpr, "FPR": fpr})

# full precision is kept in `manual`; we only round when displaying, so that the
# comparison against sklearn in section 6 and the AUC in section 7 stay exact
manual = pd.DataFrame(pts)
manual.round(4)

Walk down that table one row at a time and watch the staircase form. Each step admits exactly one more applicant:

- If that applicant **actually defaulted**, TP goes up by one and the curve steps **up**.
- If that applicant **actually paid**, FP goes up by one and the curve steps **right**.

Up for a hit, right for a false alarm. Six ups and six rights, in the order the model ranked them. **The ROC curve is a picture of the model's ranking.**

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.step(manual["FPR"], manual["TPR"], where="post", color=GOLD, lw=2.5)
ax.plot(manual["FPR"], manual["TPR"], "o", color="#3D2E00", markersize=5)
ax.plot([0, 1], [0, 1], "k--", lw=1)

ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("The staircase: up for a real defaulter, right for a false alarm")
ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Look for the one step to the **right** that happens early, at FPR = 1/6 while TPR is still only 2/6. That is applicant **C** - scored 0.85, paid the loan back. One good customer, ranked above three real defaulters, and you can see them in the shape of the curve.

🔷 **The nugget:** every wiggle in an ROC curve is a specific row in your data. The curve is not abstract - you can always point at the observation that caused a step.

## 6. Does scikit-learn agree?

We built that by hand. Now let the library do it and check the numbers line up.

One wrinkle worth knowing: `roc_curve()` defaults to `drop_intermediate=True`, which quietly discards points lying on a straight run of the curve. That is a sensible optimization for plotting a curve with thousands of points, and it is why a hand-built table sometimes has more rows than sklearn's. We pass `drop_intermediate=False` so the two line up exactly.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# drop_intermediate=False keeps every point. Left at its default True, sklearn
# throws away points that sit on a straight run - fine for plotting, but then
# its table would not line up with the one we just built by hand.
fpr_sk, tpr_sk, thr_sk = roc_curve(loans["actual"], loans["score"],
                                   drop_intermediate=False)

sk = pd.DataFrame({"threshold": thr_sk.round(4), "FPR": fpr_sk.round(4), "TPR": tpr_sk.round(4)})
print("scikit-learn's roc_curve():")
sk

In [ ]:
# Compare our hand-built points against sklearn's, ignoring row order
ours = set(zip(manual["FPR"].round(9), manual["TPR"].round(9)))
theirs = set(zip(np.round(fpr_sk, 9), np.round(tpr_sk, 9)))

print("Our points match scikit-learn's exactly:", ours == theirs)
print()
print("points we found      :", len(ours))
print("points sklearn found :", len(theirs))

## 7. AUC - and what the number actually means

**AUC** is the area under that staircase. One number, between 0 and 1, summarizing the model across **every** threshold at once - which is exactly why it is useful when you have not yet decided what threshold to use.

Compute it the boring way first: the trapezoid rule.

In [ ]:
auc_manual = np.trapezoid(manual["TPR"], manual["FPR"]) if hasattr(np, "trapezoid") \
             else np.trapz(manual["TPR"], manual["FPR"])

print(f"AUC by area under our staircase : {auc_manual:.4f}")
print(f"AUC from sklearn                : {roc_auc_score(loans['actual'], loans['score']):.4f}")

### The interpretation worth memorizing

AUC has a meaning that has nothing to do with area, and it is the one to carry with you:

> **AUC is the probability that a randomly chosen positive gets a higher score than a randomly chosen negative.**

With 6 defaulters and 6 non-defaulters there are 36 possible pairs. Count how many the model ranked correctly.

In [ ]:
positives = loans.loc[loans["actual"] == 1, "score"].to_numpy()
negatives = loans.loc[loans["actual"] == 0, "score"].to_numpy()

wins = 0
ties = 0
for p in positives:
    for n in negatives:
        if p > n:
            wins += 1
        elif p == n:
            ties += 1

total_pairs = len(positives) * len(negatives)
auc_pairs = (wins + 0.5 * ties) / total_pairs

print(f"Pairs checked          : {total_pairs}")
print(f"Ranked correctly       : {wins}")
print(f"AUC from counting pairs: {auc_pairs:.4f}")
print()
print("Same number as the area under the curve:",
      np.isclose(auc_pairs, roc_auc_score(loans["actual"], loans["score"])))

So an AUC of about 0.81 means: **pick a defaulter and a non-defaulter at random, and roughly 81% of the time the model scores the defaulter higher.**

| AUC | What it means |
| :-- | :-- |
| 0.50 | Coin flip. The scores carry no information. |
| 0.70 | Weak but real separation. |
| 0.80 | Decent. Where a lot of honest business models actually live. |
| 0.90+ | Strong - and worth checking for leakage before you celebrate. |
| 1.00 | Perfect separation. In practice, almost always a bug. |

**Caution:** AUC says nothing about whether your *probabilities* are accurate, only whether the *ranking* is good. A model that outputs 0.99 for everyone who defaults and 0.98 for everyone who doesn't has an AUC of 1.0 and probability estimates that are complete nonsense. Ranking and calibration are different things.

## 8. Choosing the threshold with money

AUC compares models. It does **not** pick your threshold. Nothing about the curve tells you where to stand on it - that comes from what the two mistakes cost.

Say a false negative (approving someone who defaults) costs you **\$10,000** in lost principal, and a false positive (turning away a good customer) costs you **\$1,000** in lost business. Missing a defaulter is ten times worse. Now the arithmetic has an opinion.

In [ ]:
COST_FN = 10_000   # approved someone who defaulted
COST_FP =  1_000   # turned away a good customer

rows = []
for t in sorted(set(loans["score"]).union({1.01}), reverse=True):
    tp, fp, fn, tn = confusion_at(loans, t)
    rows.append({"threshold": round(t, 2), "FP": fp, "FN": fn,
                 "total cost": fp * COST_FP + fn * COST_FN})

costs = pd.DataFrame(rows)
best = costs.loc[costs["total cost"].idxmin()]

# what the out-of-the-box 0.50 threshold would have cost us
tp5, fp5, fn5, tn5 = confusion_at(loans, 0.50)
cost_default = fp5 * COST_FP + fn5 * COST_FN

print(f"Cheapest threshold : {best['threshold']}   ->  ${best['total cost']:,.0f}")
print(f"Default threshold  : 0.50   ->  ${cost_default:,.0f}")
print(f"Money left on the table by not thinking about it: "
      f"${cost_default - best['total cost']:,.0f}")
print()
costs

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(costs["threshold"], costs["total cost"], "o-", color=GOLD, lw=2)
ax.axvline(best["threshold"], color="#3D2E00", ls="--", lw=1.5,
           label=f"cheapest: t={best['threshold']}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Total cost of mistakes ($)")
ax.set_title("The threshold is a business decision, and it has a price tag")
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.show()

When a miss costs ten times a false alarm, the cheapest policy is **aggressive** - a low threshold, flagging lots of people, eating false positives to avoid the expensive misses.

Flip the costs and the answer flips with them. **Nothing about the model changed.** Same predictions, same ROC curve, same AUC - a different business context and a different right answer.

🔷 **The nugget:** AUC picks your *model*. Costs pick your *threshold*. Two different questions, and conflating them is how people end up defending 0.5 as though it were a law of nature.

## 9. Where ROC misleads you: rare positives

One caveat, and it matters more than students expect.

FPR has **all the negatives** in its denominator. When negatives massively outnumber positives, a large *number* of false positives can still be a small *rate* - so the ROC curve looks reassuring while the model is unusable in practice.

Build a lopsided problem and watch it happen.

In [ ]:
from sklearn.metrics import precision_score, average_precision_score

rng = np.random.default_rng(5509)

# 10,000 transactions, 1% fraudulent - a realistic imbalance
n = 10_000
y_rare = (rng.random(n) < 0.01).astype(int)

# a decent-but-not-magic scorer: frauds score higher on average
scores_rare = np.clip(rng.normal(np.where(y_rare == 1, 0.68, 0.42), 0.16), 0, 1)

print(f"Fraud cases: {y_rare.sum()} out of {n}  ({y_rare.mean():.1%})")
print(f"AUC: {roc_auc_score(y_rare, scores_rare):.3f}   <- looks great")

In [ ]:
# Now look at what actually lands on the fraud team's desk at a sensible threshold
t = 0.60
flagged = scores_rare >= t
tp = int(((y_rare == 1) & flagged).sum())
fp = int(((y_rare == 0) & flagged).sum())

print(f"Threshold {t}")
print(f"  alerts raised      : {tp + fp:,}")
print(f"  actually fraud     : {tp:,}")
print(f"  false alarms       : {fp:,}")
print()
print(f"  FPR  = {fp / (y_rare == 0).sum():.3f}   <- small! the ROC curve is happy")
print(f"  Precision = {tp / (tp + fp):.3f}   <- of everything we flagged, this share was real")

A false positive **rate** of about 0.13 sounds tolerable. But there are nearly 9,900 legitimate transactions, so that rate produces **over 1,300 false alarms to surface fewer than 80 real frauds** - and a human being has to review every one of them. Precision lands near **0.06**: roughly 1 in 18 alerts is worth opening.

**When positives are rare, report precision and recall alongside AUC.** The precision-recall curve and its summary, **average precision**, are built for exactly this situation because neither one puts the huge negative class in a denominator.

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, _ = precision_recall_curve(y_rare, scores_rare)
fpr_r, tpr_r, _ = roc_curve(y_rare, scores_rare)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr_r, tpr_r, color=GOLD, lw=2)
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title(f"ROC - looks strong (AUC = {roc_auc_score(y_rare, scores_rare):.3f})")

axes[1].plot(rec, prec, color=GOLD, lw=2)
axes[1].axhline(y_rare.mean(), color="k", ls="--", lw=1,
                label=f"random guessing = {y_rare.mean():.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_ylim(0, 1)
axes[1].set_title(f"Precision-Recall - the honest view "
                  f"(AP = {average_precision_score(y_rare, scores_rare):.3f})")
axes[1].legend()

plt.suptitle("Same model, same predictions, two very different impressions")
plt.tight_layout()
plt.show()

## What you should have after this appendix

- You can build an ROC curve from a confusion matrix table without calling a library
- You know each point on the curve is one threshold, and each step is one observation - up for a hit, right for a false alarm
- You can explain what raising or lowering the threshold does to **TPR and FPR at the same time**, and why you cannot improve one for free
- You can define AUC two ways: area under the curve, and the probability a random positive outranks a random negative
- You know AUC judges **ranking**, not calibration, and not your choice of threshold
- You can pick a threshold from the **cost** of each mistake rather than defaulting to 0.5
- You know that with rare positives, ROC flatters a model and precision-recall tells the truth

**On your own:**

1. Change applicant **C**'s score in section 1 from `0.85` to `0.35` - so the model now ranks that good customer correctly. Re-run. What happens to the staircase, and to AUC? Explain the change by pointing at the specific step that moved.
2. In section 8, swap the costs so a false positive costs \$10,000 and a false negative costs \$1,000. Which threshold wins now? Say why in one sentence.
3. Take the model from `3_AllTheModels_Classification.ipynb`, get its `predict_proba()` output, and find the threshold that maximizes the F1 score. Is it 0.5?

---

**Where this goes next:** in Module 2 the probability comes out of a **sigmoid** on the last layer of a neural network instead of out of `predict_proba()`. The number arrives from a completely different machine. Everything in this notebook - the threshold, the confusion matrix, the ROC curve, the cost argument - applies unchanged.